In [ ]:
path = 'HCMST 2017 fresh sample for public sharing draft v1.1.dta'
import pandas as pd
data = pd.read_stata(path)

## Data preparation

In [ ]:
data.head()

In [ ]:
data.columns

In [ ]:
y = data['S1']
y.value_counts()

In [ ]:
x = data.drop(['S1'], inplace=False, axis=1)
x.head()

### Choosing ge

In [ ]:
features = ["Q9", "Q16", "w6_sex_frequency", "partner_yrsed", "ppagecat"]

* Q9 - partner's age
* Q16 - relatives seen
* partnet_yrsed - partner's educational attainment
* ppagecat - age

In [ ]:
x = x[features]

In [ ]:
x = x.rename(index=str, columns={
    "Q9": "partner_age",
    "Q16": "relatives_seen",
    "w6_sex_frequency": "sex_frequency",
    "partner_yrsed": "educational_attainment",
    "ppagecat": "age"
})

In [ ]:
x.head()

In [ ]:
for f in x.columns:
    print("\n################ {} ##########\n".format(f))
    print(x[f].value_counts(dropna=False))

In [ ]:
x = x[-pd.isna(x['educational_attainment'])]

In [ ]:
x = x[-pd.isna(x['partner_age'])]

In [ ]:
x = x[x['partner_age'] != 'Refused']

In [ ]:
x = x[-pd.isna(x['relatives_seen'])]

In [ ]:
x = x[-pd.isna(x['sex_frequency'])]

In [ ]:
x = x[x['sex_frequency'] != 'Refused']

In [ ]:
len(x)

In [ ]:
x.head()

In [ ]:
x['partner_age'] = x['partner_age'].astype('float')

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# We create the preprocessing pipelines for both numeric and categorical data.
numeric_features = ['educational_attainment', 'relatives_seen', 'partner_age']
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())])

categorical_features = ['age', 'sex_frequency']
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)])

### Train model

In [ ]:
from xgboost import XGBClassifier

In [ ]:
x.head()

In [ ]:
valid_rows = [int(row) for row in x.index]
y = y[valid_rows]

In [ ]:
print(x.shape, y.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
rf_clf = Pipeline(steps=[('preprocessor', preprocessor), ('clf', RandomForestClassifier())])
rf_clf = rf_clf.fit(x, y)

### Observation to be explained

In [ ]:
observation = x.iloc[70:71]
observation

### Person details
Widzimy osob? w przedziale wiekowym 25-34 lata, kt?rej partner ma lat 29, a jego najwy?sze wykszta?cenie to licencjat. Para uprawia seks 1-2 razy w tygodniu. Badana osoba spotyka ?rednio miesi?czne troje krewnych.

In [ ]:
rf_clf.classes_

In [ ]:
print('Szansa pozostania w zwi?zku wynosi {}'.format(rf_clf.predict_proba(observation)[0][1]))

In [ ]:
import numpy as np
def break_down(obs, variables, data, model):
    data = data.copy()
    average = np.mean(model.predict_proba(data)[::, 1])
    attributions = [average]
    for var in variables:
        data[var] = obs[var].iloc[0]
        prev_average = average
        average = np.mean(model.predict_proba(data)[::, 1])
        attributions.append(average - prev_average)
    return attributions

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
import sklearn
warnings.filterwarnings("ignore", category=sklearn.exceptions.DataConversionWarning)
def plot_break_down(obs, variables, data, model):
    plt.figure(figsize=(12, 4))
    plt.bar(['intercept'] + variables, break_down(obs, variables, data, model))    

### Generate perturbated data, define proximity

In [ ]:
observation

In [ ]:
x['sex_frequency'].value_counts()

In [ ]:
sex_freq_close_values = {
    0: 'Once a month or less',
    1: '2 to 3 times a month',
    2: 'Once or twice a week',
    3: '3 to 6 times a week',
    4: 'Once a day or more'
}
inv_sex_freq = {v: k for k, v in sex_freq_close_values.items()}

In [ ]:
x['age'].value_counts()

In [ ]:
age_close = {
    20:'18-24',
    30:'25-34',
    40:'35-44',
    50:'45-54',
    60:'55-64',
    70:'65-74',
    80:'75+'
}
inv_age = {v: k for k, v in age_close.items()}

In [ ]:
def perturbated_data(observation, n=20):
    data = pd.concat([observation for _ in range(n)])
    data.reset_index(drop=True, inplace=True)
    # how many variables we change
    changes = np.random.randint(1, observation.shape[1] + 1, n)
    weights = 1 / changes
    for i in range(n):
        changed_vars = np.random.choice(observation.columns, changes[i], replace=False)
        if 'partner_age' in changed_vars:
            data.loc[i, 'partner_age'] += np.random.choice([-5, -2, -1, 0, 1, 2, 5])
        if 'relatives_seen' in changed_vars:
            data.loc[i, 'relatives_seen'] += np.random.choice([-1, 0, 1])
        if 'educational_attainment' in changed_vars:
            data.loc[i, 'educational_attainment'] += np.random.choice([-1, 0, 1])
        if 'age' in changed_vars:
            current = data.loc[i, 'age']
            index = inv_age[current]
            index += np.random.choice([-10, 0, 10])
            data.loc[i, 'age'] = age_close.get(index, current)
        if 'sex_frequency' in changed_vars:
            current = data.loc[i, 'sex_frequency']
            index = inv_sex_freq[current]
            index += np.random.choice([-1, 0, 1])
            data.loc[i, 'sex_frequency'] = sex_freq_close_values.get(index, current)
    return data, weights

perturbated_data(observation)

### Fit decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
local_data, weights = perturbated_data(observation, n=500)
local_y = rf_clf.predict(local_data)

##### Encode numerically categorical features

In [ ]:
local_data['sex_frequency'] = local_data['sex_frequency'].replace(inv_sex_freq)

In [ ]:
local_data['age'] = local_data['age'].replace(inv_age)

In [ ]:
local_data.head(n=20)

In [ ]:
white_box = DecisionTreeClassifier(max_leaf_nodes=8)
white_box.fit(local_data, local_y, weights)

In [ ]:
white_box.feature_importances_

In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(local_data.columns, white_box.feature_importances_)

Powy?ej widzimy wa?no?? zmiennych w drzewie decyzyjnym. W przeciwie?stwie do metod break down, nie musimy martwi? si? o kolejno?? zmiennych. Widzimy, ?e podobnie jak w analizach metod? breakdown otrzymujemy najwi?ksz? wa?no?? dla zmiennych dotycz?cych wieku partner?w. Mniej wa?na jest np. *sex_frequency*

In [ ]:
plot_break_down(observation, 
                ['educational_attainment', 'partner_age', 'relatives_seen', 'sex_frequency', 'age'],
                x, rf_clf)

In [ ]:
plot_break_down(observation, 
                ['sex_frequency', 'relatives_seen', 'age', 'educational_attainment', 'partner_age'],
                x, rf_clf)

### Visualize white box decision tree

In [ ]:
from sklearn.externals.six import StringIO  
from IPython.display import Image  
from sklearn.tree import export_graphviz
import pydotplus

dot_data = StringIO()

export_graphviz(white_box, out_file=dot_data, feature_names = observation.columns,
                class_names = white_box.classes_,
                rounded = True, proportion = False, precision = 2, filled = True)

graph = pydotplus.graph_from_dot_data(dot_data.getvalue())  
Image(graph.create_png())

Nasze drzewo ma ograniczon? liczb? li?ci, wi?c jeste?my w stanie narysowa? ca?y proces decyzyjny.
Wida? np. ?e dla os?b m?odych algorytm ?atwo (i poprawnie) podejmuje decyzje.